In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from ultralytics import YOLO
import cv2, os, time

WEIGHTS_PATH = r"D:\ANTS\runs\detect\outputs\train_filtered-2\weights\best.pt"
YAML_PATH    = r"D:\ANTS\VisDrone_Dataset\visdrone_filtered.yaml"
TRAIN_DIR    = r"D:\ANTS\runs\detect\outputs\train_filtered-2"
CSV_PATH     = r"D:\ANTS\runs\detect\outputs\train_filtered-2\results.csv"
TEST_IMG_DIR = r"D:\ANTS\VisDrone_Dataset\VisDrone2019-DET-test-dev\images"
OUTPUT_DIR   = r"D:\ANTS\task05_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASS_NAMES = {0: "pedestrian", 1: "people", 2: "car"}
model = YOLO(WEIGHTS_PATH)
print("✅ Model loaded")


# ════════════════════════════════════════
# PART 1 — Read CSV & Print Summary
# ════════════════════════════════════════

df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()
# YOLO adds spaces to column names — strip() removes them

best_epoch = df['metrics/mAP50(B)'].idxmax() + 1
# idxmax() returns the row index of the highest value
# +1 because epoch numbers start at 1 not 0

best_map50   = df['metrics/mAP50(B)'].max()
best_map5095 = df.loc[df['metrics/mAP50(B)'].idxmax(), 'metrics/mAP50-95(B)']
best_prec    = df.loc[df['metrics/mAP50(B)'].idxmax(), 'metrics/precision(B)']
best_recall  = df.loc[df['metrics/mAP50(B)'].idxmax(), 'metrics/recall(B)']
# df.loc[row, col] gets the value at a specific row and column
# We get precision/recall at the epoch where mAP was best

print("\n" + "="*50)
print("  TRAINING RESULTS FROM CSV")
print("="*50)
print(f"  Total epochs trained  : {len(df)}")
print(f"  Best epoch            : {best_epoch}")
print(f"  mAP@0.5               : {best_map50:.4f}  ({best_map50*100:.2f}%)")
print(f"  mAP@0.5:0.95          : {best_map5095:.4f}  ({best_map5095*100:.2f}%)")
print(f"  Precision             : {best_prec:.4f}  ({best_prec*100:.2f}%)")
print(f"  Recall                : {best_recall:.4f}  ({best_recall*100:.2f}%)")


# ════════════════════════════════════════
# PART 2 — Official Val Metrics
# ════════════════════════════════════════

print("\nRunning validation for official metrics...")
metrics = model.val(
    data    = YAML_PATH,
    imgsz   = 512,
    batch   = 8,
    device  = 0,
    verbose = False
)

map50   = metrics.box.map50
map5095 = metrics.box.map
mp      = metrics.box.mp
mr      = metrics.box.mr

print("\n" + "="*50)
print("  OFFICIAL VALIDATION METRICS")
print("="*50)
print(f"  mAP@0.5       : {map50:.4f}  ({map50*100:.2f}%)")
print(f"  mAP@0.5:0.95  : {map5095:.4f}  ({map5095*100:.2f}%)")
print(f"  Precision     : {mp:.4f}  ({mp*100:.2f}%)")
print(f"  Recall        : {mr:.4f}  ({mr*100:.2f}%)")

print(f"\n  Per-class mAP@0.5:")
for name, ap in zip(CLASS_NAMES.values(), metrics.box.ap50):
    print(f"    {name:15s}: {ap:.4f}  ({ap*100:.2f}%)")


# ════════════════════════════════════════
# PART 3 — FPS Benchmark
# ════════════════════════════════════════

print("\nMeasuring FPS...")

test_imgs = sorted([f for f in os.listdir(TEST_IMG_DIR)
                    if f.endswith('.jpg')])[:100]

# Warmup runs — GPU needs a few inferences to reach full speed
for _ in range(3):
    model(os.path.join(TEST_IMG_DIR, test_imgs[0]), verbose=False)

start = time.time()
for img_file in test_imgs:
    model(os.path.join(TEST_IMG_DIR, img_file), conf=0.25, verbose=False)
end   = time.time()

total_time = end - start
fps        = len(test_imgs) / total_time

print(f"  Average FPS     : {fps:.2f}")
print(f"  Time per image  : {total_time/len(test_imgs)*1000:.1f}ms")


# ════════════════════════════════════════
# PART 4 — Training Curves from CSV
# ════════════════════════════════════════

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Training Curves — YOLOv8s on VisDrone',
             fontsize=14, fontweight='bold')

epochs = df['epoch']

# ── Plot 1: mAP over epochs ──
ax = axes[0, 0]
ax.plot(epochs, df['metrics/mAP50(B)'],
        color='#2ecc71', linewidth=2, label='mAP@0.5')
ax.plot(epochs, df['metrics/mAP50-95(B)'],
        color='#27ae60', linewidth=2, linestyle='--', label='mAP@0.5:0.95')
# Mark best epoch with a dot
ax.axvline(best_epoch, color='red', linestyle=':', linewidth=1,
           label=f'Best epoch: {best_epoch}')
ax.set_title('mAP over Epochs', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('mAP')
ax.legend()
ax.grid(True, alpha=0.3)

# ── Plot 2: Precision & Recall ──
ax = axes[0, 1]
ax.plot(epochs, df['metrics/precision(B)'],
        color='#3498db', linewidth=2, label='Precision')
ax.plot(epochs, df['metrics/recall(B)'],
        color='#e74c3c', linewidth=2, label='Recall')
ax.set_title('Precision & Recall over Epochs', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Value')
ax.legend()
ax.grid(True, alpha=0.3)

# ── Plot 3: Training Loss ──
ax = axes[1, 0]
ax.plot(epochs, df['train/box_loss'],
        color='#e67e22', linewidth=2, label='Box Loss')
ax.plot(epochs, df['train/cls_loss'],
        color='#9b59b6', linewidth=2, label='Class Loss')
ax.plot(epochs, df['train/dfl_loss'],
        color='#1abc9c', linewidth=2, label='DFL Loss')
# Box loss = how accurate the bounding box positions are
# Class loss = how well it predicts the correct class
# DFL loss = distribution focal loss (box precision)
ax.set_title('Training Loss over Epochs', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# ── Plot 4: Validation Loss ──
ax = axes[1, 1]
ax.plot(epochs, df['val/box_loss'],
        color='#e67e22', linewidth=2, label='Val Box Loss')
ax.plot(epochs, df['val/cls_loss'],
        color='#9b59b6', linewidth=2, label='Val Class Loss')
ax.plot(epochs, df['val/dfl_loss'],
        color='#1abc9c', linewidth=2, label='Val DFL Loss')
ax.set_title('Validation Loss over Epochs', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"),
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Training curves saved")


# ════════════════════════════════════════
# PART 5 — Confusion Matrix
# ════════════════════════════════════════

conf_path = os.path.join(TRAIN_DIR, 'confusion_matrix_normalized.png')
if os.path.exists(conf_path):
    plt.figure(figsize=(10, 8))
    plt.imshow(mpimg.imread(conf_path))
    plt.axis('off')
    plt.title('Confusion Matrix (Normalized)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"),
                dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Confusion matrix saved")


# ════════════════════════════════════════
# PART 6 — Sample Validation Predictions
# ════════════════════════════════════════

val_preds = sorted([f for f in os.listdir(TRAIN_DIR)
                    if f.startswith('val_batch') and f.endswith('.jpg')])

if val_preds:
    fig, axes = plt.subplots(1, min(3, len(val_preds)), figsize=(18, 7))
    if len(val_preds) == 1:
        axes = [axes]
    for ax, pred_file in zip(axes, val_preds[:3]):
        ax.imshow(mpimg.imread(os.path.join(TRAIN_DIR, pred_file)))
        ax.axis('off')
        ax.set_title(pred_file, fontsize=8)
    plt.suptitle('Sample Validation Predictions',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "val_predictions.png"),
                dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Validation predictions saved")


# ════════════════════════════════════════
# PART 7 — Counting Visualization
# ════════════════════════════════════════

print("\nGenerating counting visualization...")

sample_imgs = sorted([f for f in os.listdir(TEST_IMG_DIR)
                      if f.endswith('.jpg')])[:20]

fig, axes = plt.subplots(4, 5, figsize=(20, 16))
fig.suptitle('Human & Car Detection + Counting Results',
             fontsize=14, fontweight='bold')
axes_flat = axes.flatten()

for idx, img_file in enumerate(sample_imgs):
    img_path    = os.path.join(TEST_IMG_DIR, img_file)
    frame       = cv2.imread(img_path)
    result      = model(img_path, conf=0.25, verbose=False)[0]
    human_count = 0
    car_count   = 0

    if result.boxes is not None:
        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
            cls   = int(box.cls[0].item())
            color = {0:(0,255,80), 1:(0,180,50), 2:(255,100,0)}.get(cls,(255,255,0))
            cv2.rectangle(frame, (x1,y1), (x2,y2), color, 1)
            if cls in [0, 1]:
                human_count += 1
            elif cls == 2:
                car_count += 1

    cv2.rectangle(frame, (0,0), (frame.shape[1], 36), (0,0,0), -1)
    cv2.putText(frame, f"Humans: {human_count}  Cars: {car_count}",
                (8, 25), cv2.FONT_HERSHEY_SIMPLEX,
                0.7, (0,255,100), 2, cv2.LINE_AA)

    axes_flat[idx].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    axes_flat[idx].set_title(f'H:{human_count} | C:{car_count}', fontsize=8)
    axes_flat[idx].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "counting_visualization.png"),
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Counting visualization saved")


# ════════════════════════════════════════
# FINAL SUMMARY
# ════════════════════════════════════════

print("\n" + "="*50)
print("  FINAL SUMMARY")
print("="*50)
print(f"  Epochs trained  : {len(df)}")
print(f"  Best epoch      : {best_epoch}")
print(f"  mAP@0.5         : {map50*100:.2f}%")
print(f"  mAP@0.5:0.95    : {map5095*100:.2f}%")
print(f"  Precision       : {mp*100:.2f}%")
print(f"  Recall          : {mr*100:.2f}%")
print(f"  FPS             : {fps:.2f}")
print(f"\n  Strengths:")
print(f"    - {map50*100:.1f}% mAP@0.5 on challenging drone imagery")
print(f"    - Real-time capable at {fps:.1f} FPS on RTX 3050")
print(f"    - Stable training with consistent loss reduction over 90 epochs")
print(f"\n  Limitations:")
print(f"    - Small objects at high altitude remain challenging")
print(f"    - Heavy crowd occlusion reduces recall in dense scenes")
print(f"    - People class harder to detect than individual pedestrians")
print(f"\n✅ Task-05 Complete! Outputs saved to: {OUTPUT_DIR}")

✅ Model loaded

  TRAINING RESULTS FROM CSV
  Total epochs trained  : 90
  Best epoch            : 88
  mAP@0.5               : 0.4810  (48.10%)
  mAP@0.5:0.95          : 0.2524  (25.24%)
  Precision             : 0.6305  (63.05%)
  Recall                : 0.4619  (46.19%)

Running validation for official metrics...
Ultralytics 8.4.51  Python-3.10.20 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
Model summary (fused): 73 layers, 11,126,745 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.50.2 ms, read: 11.13.5 MB/s, size: 143.7 KB)
val: Scanning D:\ANTS\VisDrone_Dataset\VisDrone2019-DET-val\labels.cache... 548 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 548/548  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 69/69 5.1it/s 13.4s0.1s
                   all        548      28033      0.633      0.463      0.482      0.254
Speed: 2.0ms preprocess, 7.7ms inference, 0.0ms l

<Figure size 1400x1000 with 4 Axes>

✅ Training curves saved

Generating counting visualization...


<Figure size 2000x1600 with 20 Axes>

✅ Counting visualization saved

  FINAL SUMMARY
  Epochs trained  : 90
  Best epoch      : 88
  mAP@0.5         : 48.21%
  mAP@0.5:0.95    : 25.37%
  Precision       : 63.35%
  Recall          : 46.26%
  FPS             : 22.64

  Strengths:
    - 48.2% mAP@0.5 on challenging drone imagery
    - Real-time capable at 22.6 FPS on RTX 3050
    - Stable training with consistent loss reduction over 90 epochs

  Limitations:
    - Small objects at high altitude remain challenging
    - Heavy crowd occlusion reduces recall in dense scenes
    - People class harder to detect than individual pedestrians

✅ Task-05 Complete! Outputs saved to: D:\ANTS\task05_outputs
